# CadQuery para Arquitetos
## Notebook 1 — Introdução e Conceitos Básicos

---

### O que é o CadQuery?

**CadQuery** é uma biblioteca Python de modelagem geométrica 3D baseada em código. Diferente de softwares como Rhino ou AutoCAD, onde você modela arrastando e clicando, no CadQuery você **descreve** a geometria por meio de instruções escritas.

Essa abordagem traz vantagens importantes para arquitetos e projetistas:

- **Parametrização total**: qualquer dimensão pode ser uma variável, facilitando variações do projeto
- **Reprodutibilidade**: o código é a documentação do modelo
- **Automação**: é possível gerar dezenas de variações automaticamente
- **Integração**: o modelo pode ser conectado a planilhas, bancos de dados e outras ferramentas

O CadQuery é construído sobre o núcleo geométrico **OCCT (Open Cascade Technology)**, o mesmo motor usado pelo FreeCAD. Ele trabalha com geometria sólida real (B-Rep), o que significa que os modelos gerados têm volume, massa e propriedades físicas precisas.

---

### Visualização no VS Code

Neste curso usamos a extensão **OCP CAD Viewer** no VS Code para visualizar os modelos 3D diretamente no editor, sem precisar abrir nenhum software externo.

> ⚠️ **Antes de começar**: certifique-se de que a extensão OCP CAD Viewer está ativa no VS Code. Você verá o painel do visualizador na lateral ou em uma aba separada.

---

## Importações

In [1]:
import cadquery as cq
from ocp_vscode import show, set_port, show_object

In [2]:
set_port(3939)

---

## Primeiro Exemplo

Antes de entrar nos detalhes da biblioteca, veja como é simples criar e visualizar um objeto 3D com o CadQuery. O código abaixo cria uma caixa e a envia para o visualizador:

In [3]:
caixa = cq.Workplane("XY").box(5, 3, 2)

show(caixa)

+


Em apenas duas linhas: uma cria a geometria, a outra a exibe. Nos próximos tópicos vamos entender o que cada parte desse código significa.

---

## O Plano de Trabalho (Workplane)

Toda modelagem no CadQuery começa com um **Workplane** (plano de trabalho). Pense nele como a "superfície" sobre a qual você começa a construir, similar ao plano de referência de um software BIM.

O Workplane define:
- **Qual plano** do espaço 3D você está usando como base
- **A partir de onde** a geometria será construída

### Forma básica

Na forma mais simples, basta informar o nome do plano:

```python
cq.Workplane("XY")
```

Os três planos principais são:

| Nome | Plano | Uso comum |
|------|-------|-----------|
| `"XY"` | Horizontal (chão) | Plantas baixas, lajes |
| `"XZ"` | Frontal (fachada) | Elevações frontais |
| `"YZ"` | Lateral | Cortes laterais |

Nessa forma simples, a **origem do plano** coincide com o ponto `(0, 0, 0)` do espaço 3D.

---

### Definindo uma origem personalizada

Muitas vezes queremos começar a construir a partir de um ponto específico, e não da origem `(0, 0, 0)`. Para isso, usamos o parâmetro `origin`:

```python
cq.Workplane("XY", origin=(x, y, z))
```

Isso desloca o **ponto de partida** da construção para a coordenada escolhida. Tudo que for criado a partir desse Workplane terá sua posição calculada em relação a essa nova origem.

Veja a comparação: quatro caixas idênticas criadas com origens diferentes.

In [13]:
# Caixa na origem padrão (0, 0, 0)
caixa_centro = cq.Workplane("XY").box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo X
caixa_direita = cq.Workplane("XY", origin=(120, 0, 0)).box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo Y
caixa_frente = cq.Workplane("XY", origin=(0, 120, 0)).box(60, 60, 60)

# Caixa com origem elevada 60mm no eixo Z
# Útil para empilhar volumes, como pavimentos de um edifício
caixa_cima = cq.Workplane("XY", origin=(0, 0, 60)).box(60, 60, 60)

show(
    caixa_centro,
    caixa_direita,
    caixa_frente,
    caixa_cima,
    names=["Origem (0,0,0)", "X deslocado", "Y deslocado", "Z elevado"]
)

cccc


> 💡 **Dica**: usar `origin` no Workplane é especialmente útil quando você quer empilhar volumes (como lajes e pavimentos) ou alinhar elementos a uma grade de projeto.

---

## Formas Primitivas Básicas

O CadQuery oferece formas geométricas prontas chamadas **primitivas**. São o ponto de partida para construções mais complexas.

| Função | Descrição | Parâmetros principais |
|--------|-----------|----------------------|
| `.box(l, w, h)` | Caixa retangular | comprimento, largura, altura |
| `.cylinder(h, r)` | Cilindro | altura, raio |
| `.sphere(r)` | Esfera | raio |
| `.cone(h, r1, r2)` | Cone/Tronco de cone | altura, raio da base, raio do topo |

---

## Visualização: `show()` e `show_object()`

O OCP CAD Viewer oferece duas funções para enviar geometria ao visualizador. Elas têm comportamentos diferentes e cada uma é mais adequada para situações distintas.

### `show()`

A função `show()` **limpa o visualizador** e exibe o que foi passado naquela chamada. Cada execução apaga o que estava sendo mostrado antes.

Ela aceita **vários objetos ao mesmo tempo** e o parâmetro `names` (no plural) para nomear cada um na lista de objetos do visualizador:

```python
show(objeto_a, objeto_b, names=["Nome A", "Nome B"])
```

### `show_object()`

A função `show_object()` **adiciona** um objeto ao visualizador **sem apagar** o que já está lá. É útil para construir a cena aos poucos, em células separadas.

Ela recebe **um objeto por vez** e o parâmetro `name` (no singular):

```python
show_object(objeto, name="Nome do Objeto")
```

> ⚠️ **Atenção**: se você executar uma célula com `show()` depois de ter usado `show_object()`, o visualizador será limpo e mostrará apenas o conteúdo do `show()` mais recente.

---

Vamos ver a diferença na prática. Primeiro, criamos os dois objetos que usaremos nos exemplos:
- Uma **caixa** com 40mm de altura
- Um **cilindro** com 120mm de altura — bem mais alto, para que possam ser visualizados juntos sem uma forma esconder a outra

In [15]:
# Caixa com 80 x 80mm de base e 40mm de altura
caixa = cq.Workplane("XY").box(80, 80, 40)

# Cilindro com raio 25mm e altura 120mm
# Mais alto que a caixa para ficar visível acima dela
cilindro = cq.Workplane("XY").cylinder(120, 25)

### Exemplo com `show()` — tudo de uma vez, visualizador reiniciado

Execute a célula abaixo. O visualizador exibirá a caixa e o cilindro juntos. Se você executar novamente, o visualizador mostrará exatamente o mesmo resultado — sem acumular cópias.

In [16]:
# show() limpa o visualizador e exibe todos os objetos passados de uma vez
show(caixa, cilindro, names=["Caixa", "Cilindro"])

cc


### Exemplo com `show_object()` — um objeto por vez, visualizador acumulado

Execute as duas células abaixo **em sequência**. Após a primeira, apenas a caixa aparece. Após a segunda, o cilindro é **adicionado** — a caixa continua visível.

In [17]:
# Primeira chamada: adiciona apenas a caixa ao visualizador
show_object(caixa, name="Caixa")

ccc


In [18]:
# Segunda chamada: adiciona o cilindro — a caixa continua visível!
show_object(cilindro, name="Cilindro")

cccc


> 🔎 **Resumo prático**:
> - Use **`show()`** quando quiser ver o **resultado final** de uma cena completa, passando todos os objetos de uma vez
> - Use **`show_object()`** quando quiser **inspecionar objetos progressivamente**, célula por célula, sem perder o que já estava no visualizador

---

## Usando Variáveis para Parametrizar

Uma das grandes vantagens de modelar em código é usar **variáveis** para as dimensões. Mudar o projeto inteiro se torna tão simples quanto alterar um único número.

In [19]:
# Dimensões da caixa — experimente mudar e executar novamente!
caixa_comprimento = 80    # mm
caixa_largura     = 80    # mm
caixa_altura      = 40    # mm

# Dimensões do cilindro
cilindro_raio   = 25      # mm
cilindro_altura = 120     # mm

# Criando os objetos com as variáveis
caixa_param    = cq.Workplane("XY").box(caixa_comprimento, caixa_largura, caixa_altura)
cilindro_param = cq.Workplane("XY").cylinder(cilindro_altura, cilindro_raio)

show(
    caixa_param,
    cilindro_param,
    names=["Caixa", "Cilindro"]
)

cc


---

## Movendo Objetos: `.translate()`

Para posicionar um sólido no espaço após criá-lo, usamos o método `.translate()`. Ele recebe uma **tupla** com três valores: `(x, y, z)`.

- `x` → move para a direita (positivo) ou esquerda (negativo)
- `y` → move para frente (positivo) ou trás (negativo)
- `z` → move para cima (positivo) ou baixo (negativo)

> 💡 **`origin` ou `.translate()`?** O parâmetro `origin` define o ponto de partida *antes* de criar a forma — você pensa onde vai construir. O `.translate()` move a forma *depois* de criada — você pensa onde vai colocar. Para posicionamento simples os dois chegam ao mesmo resultado. Use o que tornar o código mais fácil de ler.

In [20]:
# Caixa posicionada na origem
caixa_fixa = cq.Workplane("XY").box(80, 80, 40)

# Cilindro criado na origem e depois deslocado 120mm para a direita
cilindro_mov = cq.Workplane("XY").cylinder(120, 25)
cilindro_mov = cilindro_mov.translate((120, 0, 0))

show(
    caixa_fixa,
    cilindro_mov,
    names=["Caixa", "Cilindro deslocado"]
)

c+


---

## Rotacionando Objetos: `.rotate()`

Para rotacionar um sólido, usamos `.rotate()`. Ele funciona definindo um **eixo de rotação** (descrito por dois pontos) e um **ângulo em graus**.

```python
objeto.rotate(ponto_inicio_eixo, ponto_fim_eixo, angulo)
```

Para os eixos principais:

| Eixo | Ponto inicial | Ponto final | Efeito visual |
|------|--------------|-------------|---------------|
| Z | `(0,0,0)` | `(0,0,1)` | Girar em planta |
| X | `(0,0,0)` | `(1,0,0)` | Inclinar para frente/trás |
| Y | `(0,0,0)` | `(0,1,0)` | Inclinar para os lados |

In [21]:
# Placa retangular na posição original
placa_original = cq.Workplane("XY").box(120, 20, 10)

# A mesma placa rotacionada 45° no plano horizontal (em torno do eixo Z)
placa_girada = cq.Workplane("XY").box(120, 20, 10)
placa_girada = placa_girada.rotate((0, 0, 0), (0, 0, 1), 45)

show(
    placa_original,
    placa_girada,
    names=["Original", "Rotacionada 45°"]
)

++


---

## Escalando Objetos

O CadQuery não tem um método `.scale()` diretamente acessível no Workplane. Para aplicar escala, precisamos usar o motor geométrico OCCT que está por baixo do CadQuery.

Para deixar o código claro e simples de usar no dia a dia, vamos criar uma **função auxiliar** chamada `escalar()`. Você a define uma vez e a usa sempre que precisar — sem precisar entender os detalhes internos do OCCT.

A função recebe:
- `objeto` → o objeto CadQuery a ser escalado
- `sx` → fator de escala no eixo X
- `sy` → fator de escala no eixo Y
- `sz` → fator de escala no eixo Z

Um fator `1.0` mantém a dimensão original. Um fator `2.0` dobra. Um fator `0.5` reduz à metade.

> ⚠️ **Atenção**: a escala é aplicada em relação à **origem** do objeto `(0, 0, 0)`. Se o objeto estiver centrado na origem, o resultado fica centrado também. Se estiver deslocado, o deslocamento também será escalado — por isso é recomendado escalar *antes* de transladar.

In [23]:
# Importações necessárias para a função de escala
from OCP.gp import gp_GTrsf, gp_Mat
from OCP.BRepBuilderAPI import BRepBuilderAPI_GTransform

# Função auxiliar de escala não-uniforme
# Defina esta função no início do seu projeto e use quantas vezes quiser
def escalar(objeto, sx, sy, sz):
    """
    Aplica escala independente nos eixos X, Y e Z a um objeto CadQuery.
    
    objeto : objeto CadQuery (Workplane)
    sx     : fator de escala no eixo X (1.0 = sem alteração)
    sy     : fator de escala no eixo Y (1.0 = sem alteração)
    sz     : fator de escala no eixo Z (1.0 = sem alteração)
    """
    # Cria a matriz de transformação com os fatores de escala
    matriz = gp_Mat(
        sx,  0,  0,
         0, sy,  0,
         0,  0, sz
    )

    # Aplica a transformação à geometria do objeto
    transformacao = gp_GTrsf()
    transformacao.SetVectorialPart(matriz)

    construtor = BRepBuilderAPI_GTransform(objeto.val().wrapped, transformacao, True)

    # Devolve um novo objeto CadQuery com a geometria transformada
    resultado = cq.Workplane().add(cq.Shape(construtor.Shape()))
    return resultado

Agora vamos ver a função em ação. Vamos criar uma esfera original e quatro cópias, cada uma com um tipo diferente de escala.

Usamos `import copy` para duplicar o objeto antes de escalar, garantindo que cada versão seja independente:

In [24]:
import copy

# Esfera original com raio 30mm
esfera_original = cq.Workplane("XY").sphere(30)

# Cópia 1: escala uniforme — cresce igualmente em todas as direções
# O resultado continua sendo uma esfera, porém maior
esfera_uniforme = escalar(copy.copy(esfera_original), sx=2.0, sy=2.0, sz=2.0)

# Cópia 2: escala apenas no eixo X — a esfera se estica horizontalmente
# Resultado: elipsoide achatado nos eixos Y e Z (como um disco)
esfera_escala_x = escalar(copy.copy(esfera_original), sx=3.0, sy=1.0, sz=1.0)

# Cópia 3: escala apenas no eixo Y — a esfera se estica em profundidade
esfera_escala_y = escalar(copy.copy(esfera_original), sx=1.0, sy=3.0, sz=1.0)

# Cópia 4: escala apenas no eixo Z — a esfera se alonga verticalmente
# Resultado: elipsoide alongado como um ovo
esfera_escala_z = escalar(copy.copy(esfera_original), sx=1.0, sy=1.0, sz=3.0)

# Posicionando as esferas lado a lado para comparação
# Espaçamento de 120mm entre cada uma
esfera_uniforme  = esfera_uniforme.translate(( 120, 0, 0))
esfera_escala_x  = esfera_escala_x.translate(( 240, 0, 0))
esfera_escala_y  = esfera_escala_y.translate(( 360, 0, 0))
esfera_escala_z  = esfera_escala_z.translate(( 480, 0, 0))

show(
    esfera_original,
    esfera_uniforme,
    esfera_escala_x,
    esfera_escala_y,
    esfera_escala_z,
    names=[
        "Original",
        "Uniforme (x2 em XYZ)",
        "Escala em X (x3)",
        "Escala em Y (x3)",
        "Escala em Z (x3)"
    ]
)

+++++


> 🔎 **Observe no visualizador**: a esfera original permanece esférica. A cópia com escala uniforme também é uma esfera, porém maior. As três últimas se deformam em elipsoides — cada uma alongada em uma direção diferente. Esse tipo de deformação é útil para modelar volumes orgânicos, cúpulas achatadas ou qualquer forma que derive de uma esfera, mas com proporções distintas nos eixos.

---

## Exportando o Modelo

O CadQuery permite exportar os modelos para diferentes formatos:

| Formato | Extensão | Uso |
|---------|----------|-----|
| STEP | `.step` | Troca com outros softwares CAD (Rhino, FreeCAD, Fusion 360) |
| STL  | `.stl`  | Impressão 3D, renderização |
| DXF  | `.dxf`  | Compatível com AutoCAD |

In [25]:
# Criando um modelo para exportar
modelo_exportar = cq.Workplane("XY").box(100, 80, 50)

# Exportando como STEP (recomendado para troca com outros softwares)
cq.exporters.export(modelo_exportar, "meu_modelo.step")

# Exportando como STL (para impressão 3D)
cq.exporters.export(modelo_exportar, "meu_modelo.stl")

print("Arquivos exportados com sucesso!")
print("Os arquivos foram salvos na mesma pasta deste notebook.")

Arquivos exportados com sucesso!
Os arquivos foram salvos na mesma pasta deste notebook.


---

## Exercício

Crie um modelo simples que represente **uma composição de três volumes arquitetônicos**: uma torre cilíndrica central e dois blocos retangulares nos lados.

**Requisitos:**
1. A torre cilíndrica deve ser visivelmente mais alta que os blocos
2. Use `origin` no Workplane para posicionar pelo menos um dos volumes
3. Use `.translate()` para posicionar pelo menos um dos volumes
4. Use variáveis para todas as dimensões
5. Exiba tudo com `show()` usando `names`
6. **Desafio**: exiba os três objetos individualmente com `show_object()`, em células separadas

In [ ]:
# Escreva seu código aqui



---

## Resumo

Neste notebook você aprendeu:

- O que é o CadQuery e por que é útil para arquitetos
- O conceito de **Workplane** e como definir uma **origem personalizada** com `origin=(x, y, z)`
- As formas primitivas: `box`, `cylinder`, `sphere`, `cone`
- A diferença entre **`show()`** (limpa e exibe tudo de uma vez) e **`show_object()`** (adiciona ao visualizador progressivamente)
- Como usar **variáveis** para parametrizar o modelo
- Como **posicionar** objetos com `.translate()`
- Como **rotacionar** objetos com `.rotate()`
- Como **escalar** objetos com a função auxiliar `escalar()`, tanto de forma uniforme quanto por eixo individual
- Como **exportar** modelos para STEP e STL

No próximo notebook veremos como realizar **operações booleanas** — união, subtração e interseção — para criar formas mais complexas a partir da combinação de sólidos simples.

---
*CadQuery para Arquitetos — Notebook 1*